# 05 D* Lite：从目标往回的增量最短路


本课按百科条目写。D\* Lite 是「地图会变」时的增量最短路，不是又一种 A\* 启发。调度台在最后。

---

## 1. 定义

**D\* Lite**（Koenig & Likhachev, AAAI 2002 / 后续期刊）在 **边权变化** 后重修最短路，尽量复用上一次的 g 值。机器人向目标走、前方突然堵住，是典型场景。

名字来自 Stentz 的 D\*（Dynamic A\*）；Lite 用一套更简单的 rhs/g 维护，效果同类。

它从 **目标** 向全图传播「还要花多少」，类似反向 Dijkstra，但每个点留两个数，好做局部更新。

- 输入：图、当前起点 `start`、目标 `goal`。之后可多次：改边权 / 封格子，再 `compute`。
- 输出：当前 start 到 goal 的一条最短路（边权非负时）。

---

## 2. 和 A* / 普通重规划

| | 再跑一次 A\* | D\* Lite |
|--|--|--|
| 搜索方向 | 从当前点向前 | 从目标向后 |
| 图变化 | 整棵搜索树作废 | 只把受影响点标成不一致，再传播 |
| 一次查询 | 更简单、常数小 | 多次重规划、变化局部时更强 |
| 实现 | 03 课就会 | 要维护 rhs、U、km |

若地图只规划一次，用 A\*。若每走一步前方都可能多一堵墙，才值得 D\* Lite。

---

## 3. 不变量：g 与 rhs

每个点 u：

- \(g(u)\)：目前记下的、u 到目标的代价（可能过期）。
- \(rhs(u)=\min_{v\in succs(u)}\big(c(u,v)+g(v)\big)\)。目标规定 \(rhs(goal)=0\)。

含义：rhs 是「假如下一步走到某个后继，再用那个后继已记下的 g」的一拍前瞻。

| 关系 | 名称 | 算法做什么 |
|------|------|------------|
| \(g=rhs\) | 局部一致 | 这个点的值可信 |
| \(g>rhs\) | 过一致 over-consistent | 发现了更好的，把 g 降到 rhs，再通知前驱 |
| \(g<rhs\) | 欠一致 under-consistent | 某条便宜边没了，把 g 抬到 ∞ 再重算 |

优先队列 U 里是 **不一致** 的点。键是二元组（教学版用列表排序）：

\[
k(u)=\big(\min(g,rhs)+h(start,u)+km,\; \min(g,rhs)\big)
\]

\(h(start,u)\) 把更新从当前机器人位置「拉」过来。\(km\) 是机器人移动后的启发修正，避免每次重排整个堆。

**主循环停：** U 空，或堆顶键不优于 `Key(start)`，且 start 已局部一致。此时从 start 每步走到使 \(c+g(v)\) 最小的邻居，得到最短路。

---

## 4. 伪代码（对照下面 MiniDStar 与库 `DStarLite`）

```
rhs[goal] ← 0;  g 其余视为 ∞
U.insert(goal, Key(goal))

ComputeShortestPath:
    while U 非空 且 (U.TopKey < Key(start) 或 rhs[start] ≠ g[start]):
        u ← U.pop()
        if g[u] > rhs[u]:          # 过一致：变好了
            g[u] ← rhs[u]
            对每个前驱 s: UpdateVertex(s)
        else:                      # 欠一致：变坏了
            g[u] ← ∞
            UpdateVertex(u) 以及每个前驱

UpdateVertex(u):
    若 u ≠ goal: rhs[u] ← min_v c(u,v)+g[v]
    从 U 删掉 u
    若 g[u] ≠ rhs[u]: U.insert(u, Key(u))

边/格变化后:
    km ← km + h(last, start)
    UpdateVertex(受影响点)
    ComputeShortestPath
```

网格无向：前驱 = 邻居。有向图必须扫「谁指向 u」，否则更新会漏。

教学实现的 U 用列表排序，最坏较慢；论文用堆。正确性不依赖堆实现，只依赖键的顺序。

---

## 5. 手算（三个点的线）

格子 `S=A=G` 横排，边权 1：

```
S --1-- A --1-- G
```

从 G 往回：

1. 插入 G，rhs(G)=0，g(G)=∞ → 过一致。弹出 G，g(G)←0，通知 A。
2. rhs(A)=c(A,G)+g(G)=1，g(A)=∞ → 弹出 A，g(A)←1，通知 S。
3. rhs(S)=1+g(A)=2，弹出 S，g(S)←2。

提取：S 选使 c+g 最小的邻居 → A → G。路长 2。

现在封掉 A–G（或把 A 变成墙，S 只能……本课下一步用「把中间格设成障碍」）。rhs(A) 变成 ∞（没有后继到 G），A 欠一致，g(A) 被抬起来，S 跟着变。若图上另有绕路，rhs 会改走那条。


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "lib").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))


## 6. 实现：最小 D* Lite（无向网格）

先在 **一行三格** 上把 rhs/g 打出来。障碍用 `blocked` 集合；邻居函数不返回障碍。


In [ ]:
from lib.pretty import draw_grid

# 一行三格：S=(0,0) A=(1,0) G=(2,0)。稍后把 A 封掉。
CELLS = [(0, 0), (1, 0), (2, 0)]
START, GOAL = (0, 0), (2, 0)
blocked = set()


def nbrs(u):
    x, y = u
    out = []
    for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
        v = (x + dx, y + dy)
        if v in CELLS and v not in blocked:
            out.append((v, 1.0))
    return out


def h(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


class MiniDStar:
    def __init__(self, start, goal):
        self.start = start
        self.goal = goal
        self.g = {}
        self.rhs = {}
        self.U = []
        self.km = 0.0
        self.last = start
        self.rhs[goal] = 0.0
        self._insert(goal)

    def _key(self, u):
        gg = min(self.g.get(u, float("inf")), self.rhs.get(u, float("inf")))
        return (gg + h(self.start, u) + self.km, gg)

    def _insert(self, u):
        self.U = [(k, x) for k, x in self.U if x != u]
        self.U.append((self._key(u), u))
        self.U.sort(key=lambda item: item[0])

    def update_vertex(self, u):
        if u != self.goal:
            best = float("inf")
            for v, c in nbrs(u):
                best = min(best, c + self.g.get(v, float("inf")))
            self.rhs[u] = best
        self.U = [(k, x) for k, x in self.U if x != u]
        if self.g.get(u, float("inf")) != self.rhs.get(u, float("inf")):
            self._insert(u)

    def compute(self, verbose=True):
        steps = 0
        while self.U and (
            self.U[0][0] < self._key(self.start)
            or self.rhs.get(self.start, float("inf")) != self.g.get(self.start, float("inf"))
        ):
            _k, u = self.U.pop(0)
            steps += 1
            gu, ru = self.g.get(u, float("inf")), self.rhs.get(u, float("inf"))
            if verbose:
                print(f"弹出 {u}  g={gu} rhs={ru}  {'过一致' if gu > ru else '欠一致或其它'}")
            if gu > ru:
                self.g[u] = ru
                # 无向：前驱=邻居
                for s, _c in nbrs(u):
                    self.update_vertex(s)
            else:
                self.g[u] = float("inf")
                self.update_vertex(u)
                for s, _c in nbrs(u):
                    self.update_vertex(s)
        if verbose:
            print("compute 结束，弹出次数", steps, "g(S)=", self.g.get(self.start), "rhs(S)=", self.rhs.get(self.start))

    def extract(self):
        if self.rhs.get(self.start, float("inf")) == float("inf"):
            return []
        path = [self.start]
        cur = self.start
        seen = {cur}
        while cur != self.goal:
            best, nxt = float("inf"), None
            for v, c in nbrs(cur):
                val = c + self.g.get(v, float("inf"))
                if val < best:
                    best, nxt = val, v
            if nxt is None or nxt in seen:
                return []
            path.append(nxt)
            seen.add(nxt)
            cur = nxt
        return path


planner = MiniDStar(START, GOAL)
planner.compute()
print("静态路径", planner.extract())


对照第 5 节：应先弹出 G（过一致），再 A，再 S，路径 `[(0,0),(1,0),(2,0)]`。

## 7. 逐行

| 块 | 作用 |
|----|------|
| `rhs[goal]=0`，其余 g 缺省 ∞ | 只有终点一开始不一致 |
| `_key` 二元组 | 先比「还剩多少+h+km」，再比 min(g,rhs) |
| `update_vertex` | 重算 rhs；不一致才进 U |
| `g>rhs` 支 | 变好：采纳 rhs，向前驱传好消息 |
| else 支 | 变坏：g←∞，自己和前驱都要重算 |
| `extract` | 沿 c+g(v) 下降；`seen` 防环 |

---

## 8. 封掉中间格再 compute


In [ ]:
print("封掉中间格 A=(1,0)")
blocked.add((1, 0))
planner.km += h(planner.last, planner.start)
planner.update_vertex((0, 0))
planner.update_vertex((1, 0))
planner.update_vertex((2, 0))
planner.compute()
print("新路径（一行被切断应失败）", planner.extract())
print(draw_grid(3, 1, obstacles=blocked, start=START, goal=GOAL, path=planner.extract()))


一行被切断后没有绕路，extract 返回空。换迷宫：库实现会在封路后改走另一条，代价与重新 A\* 相同。


In [ ]:
from lib.maps import grid_maze
from lib.algos.single_agent import DStarLite
from lib.search import astar
from lib.pretty import draw_grid

g = grid_maze()
s, t = (0, 0), (7, 7)
planner = DStarLite(g, s, t)
r1 = planner.compute()
a1 = astar(g, s, t, record=False)
print("静态 D* Lite", r1.cost, "A*", a1.cost, "应相等", r1.cost == a1.cost)
print(draw_grid(g.width, g.height, obstacles=g.obstacles, start=s, goal=t, path=r1.path))

mid = r1.path[len(r1.path) // 2]
print("封掉路径中点", mid)
g.block(mid)
prev = r1.path[r1.path.index(mid) - 1]
planner.edge_blocked(prev, mid)
r2 = planner.compute()
print("新代价", r2.cost, "还经过 mid?", mid in r2.path)
print(draw_grid(g.width, g.height, obstacles=g.obstacles, start=s, goal=t, path=r2.path))


## 常见 bug

1. 封格后只调 `edge_blocked` / `update_vertex`，**忘了改图**（`g.block`）：邻居列表仍含旧边，rhs 算错。
2. 有向图用 `neighbors` 当全部前驱：反向边的点收不到「变坏」消息。
3. extract 不设 `seen`：g 尚未一致时可能转圈。
4. 把 D\* Lite 的 g 当成「从 start 出发的正向 g」。这里 g 是 **到目标** 的。
5. 机器人已经走到新格子，却不更新 `start` 和 `km`：键过期，U 顺序错。

## 条目小结

| 项目 | 内容 |
|------|------|
| 方向 | 从 goal 反向 |
| 核心 | 局部一致 g=rhs；U 装不一致点 |
| 变化 | UpdateVertex 受影响点，再 Compute |
| 何时用 | 多次重规划；一次查询用 A\* 即可 |
| 下一课 | 均匀网格上用跳点减少 A\* 扩展 |

## 练习

1. 用自己的话注释 `g>rhs` / else 两支：「过一致」和「欠一致」各对应「发现捷径」还是「捷径被拆」。
2. 静态迷宫上 D\* Lite 代价必须等于 A\*。若不等，先查 extract 是否走了 g 仍为 ∞ 的点。
3. 有向环 I0→I1→I2… 上封一条边，为什么必须扫 `graph.edges` 找前驱？


In [ ]:
from lib.studio.widget import PlannerStudio
studio = PlannerStudio()
studio.show_search({**g.to_studio(), "start": [0, 0], "goal": [7, 7]}, r2, "D* Lite")
studio
